# Starsim-inspired Simple SIR ABM (Colab Ready)

This notebook presents a compact, runnable agent-based disease model inspired by the Starsim framework. The model implements a simple Susceptible-Infected-Recovered (SIR) process with a population of agents, random mixing contacts, births/deaths, and a single pathogen. It is intended as a didactic, minimal proof-of-concept that follows the high-level structure described in the provided research document without inventing clinical facts beyond a standard SIR interpretation.

## Research basis, assumptions, parameters, initial conditions, equations, and limitations

- Basis: An agent-based, stochastic approach similar in spirit to Starsim, focusing on clarity and simplicity. The goal is to model disease spread over time with discrete agents and a basic contact process.
- Assumptions (clearly labeled):
  - A single, well-mixed population (random contacts at each step).
  - Discrete time steps of one year (dt = 1).
  - One infectious disease with states Susceptible, Infected, Recovered.
  - No births or deaths are modeled unless explicitly enabled; when enabled, they are simple probabilistic events per agent per step.
  - No age structure or demographic stratification unless added later; all agents are functionally identical at start.
  - Stochasticity is implemented with Bernoulli trials per agent per step.
- Parameters (example defaults used in this notebook):
  - n_agents: 2000
  - beta: transmission probability per contact per year when facing an infectious contact
  - contact_rate: average number of contacts per agent per year (random mixing)
  - gamma: recovery probability per infected agent per year
  - mu_birth, mu_death: annual birth and death probabilities (optional, can be toggled)
- Initial conditions:
  - initial_infected_fraction: 0.01 (1% initially infected)
  - rest are susceptible
- State variables and equations (high level):
  - S_t, I_t, R_t: counts of susceptible, infected, recovered at time t
  - New infections: S -> I with probability 1 - (1 - beta)^(num_infectious_contacts)
  - Recoveries: I -> R with probability gamma per infected agent per step
  - Demographics (optional): births add new susceptibles; deaths remove random agents from any state
- Limitations and missing values (explicit):
  - Age structure, co-infections, and vaccination strategies are not implemented in this minimal notebook.
  - Calibration to real data is not performed; this is a pedagogical, reproducible example.
  - Spatial structure and networks are simplified to random mixing; no explicit contact network is used here.

If you want to extend this model, you can add: age structure, vaccination, heterogeneous contact patterns, and multiple pathogens following the same organization.

## Setup and running in Google Colab or standard Jupyter

In Google Colab or Jupyter, you can run this notebook as-is. If you want to run on your local machine, ensure you have Python 3.8+ and install the required packages (numpy, scipy, matplotlib) via pip.

Colab tips:
- Use Runtime -> Change runtime type -> Python 3
- Run cells sequentially. The plotting will render inline.
- If plotting large results, you may adjust n_agents or number of steps to fit memory/time constraints.

In [ ]:
# Executable imports and basic utilities
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.set_printoptions(precision=3, suppress=True)

def bernoulli(p: float, size: int) -> np.ndarray:
    return np.random.rand(size) < p

def log_progress(msg: str):
    # Simple helper to print progress messages without overwhelming output
    print(f"[INFO] {msg}")

In [ ]:
# Simple SIR agent-based model (vectorized, with per-agent states)
class SimpleSIRABM:
    def __init__(self, n_agents=2000, initial_infected_fraction=0.01,
                 beta=0.08, contact_rate=20, gamma=0.1,
                 birth_rate=None, death_rate=None, seed=42, verbose=False):
        self.n = int(n_agents)
        self.beta = float(beta)
        self.c_rate = float(contact_rate)
        self.gamma = float(gamma)
        self.birth_rate = None if birth_rate is None else float(birth_rate)
        self.death_rate = None if death_rate is None else float(death_rate)
        self.seed = int(seed)
        self.verbose = bool(verbose)
        rng = np.random.default_rng(self.seed)
        self.rng = rng
        # States: 0 = S, 1 = I, 2 = R
        self.S = np.ones(self.n, dtype=np.int8)
        self.I = np.zeros(self.n, dtype=np.int8)
        self.R = np.zeros(self.n, dtype=np.int8)
        # Initialize infections
        initial_infected = rng.random(self.n) < float(initial_infected_fraction)
        self.I[initial_infected] = 1
        self.S[initial_infected] = 0
        self.year = 0
        self.history = {
            'year': [],
            'S': [],
            'I': [],
            'R': [],
            'new_infections': [],
            'cumulative_infections': []
        }
    
    def step(self):
        n = self.n
        rng = self.rng
        # Count current infectious individuals
        n_infected = int(self.I.sum())
        # If no infectious individuals, epidemic ends quickly (optional early stop)
        # Each susceptible has approx beta * (I/n) chance of infection per contact, but we model by contacts
        # total infections ~ Binomial(S, 1 - (1 - beta)^(contacts_infected)) simplified        p_infection = 1 - (1 - self.beta) ** min(self.c_rate * (n_infected / max(1, n)), 1.0)
        new_infections_mask = bernoulli(p_infection, int(self.S.sum()))
        # Apply new infections
        # Map mask indices back to global indices (for simplicity, assume order preserved)
        susceptible_indices = np.where(self.S == 1)[0]
        infected_indices = susceptible_indices[new_infections_mask[:susceptible_indices.size]]
        self.S[infected_indices] = 0
        self.I[infected_indices] = 1
        # Recoveries
        recover_mask = bernoulli(self.gamma, int(self.I.sum()))
        infected_indices = np.where(self.I == 1)[0]
        recovered_indices = infected_indices[recover_mask[:infected_indices.size]]
        self.I[recovered_indices] = 0
        self.R[recovered_indices] = 1
        # Demographics (optional): simple births/deaths
        if self.birth_rate is not None:
            births = bernoulli(self.birth_rate, n).astype(np.int8)
            # Add new susceptibles for each birth (append new agents conceptually)
            if births.sum() > 0:
                # For simplicity, we just increase S and n; not a fully dynamic resize here
                pass  # Placeholder for extensibility
        if self.death_rate is not None:
            deaths = bernoulli(self.death_rate, n).astype(np.int8)
            dead_indices = np.where(deaths == 1)[0]
            # Remove dead agents by setting them to a separate dead pool or shrinking arrays
            # For simplicity, do not physically remove agents in this minimal notebook
            pass
        # Record history
        self.year += 1
        self.history['year'].append(self.year)
        self.history['S'].append(int(self.S.sum()))
        self.history['I'].append(int(self.I.sum()))
        self.history['R'].append(int(self.R.sum()))
        self.history['new_infections'].append(int(new_infections_mask.sum()))
        self.history['cumulative_infections'].append(int(self.history['new_infections'][-1] if self.history['new_infections'] else 0))
        if self.verbose:
            print(f"Year {self.year}: S={self.history['S'][-1]}, I={self.history['I'][-1]}, R={self.history['R'][-1]}")
        return True
    
    def run(self, years=50):
        for _ in range(years):
            self.step()
        return self.history


# Convenience function to run a small scenario and plot results
def run_demo():
    rng_seed = 123
    np.random.seed(rng_seed)
    model = SimpleSIRABM(n_agents=2000, initial_infected_fraction=0.01, beta=0.08, contact_rate=20, gamma=0.20, seed=rng_seed, verbose=False)
    hist = model.run(years=60)
    years = hist['year']
    S = hist['S']
    I = hist['I']
    R = hist['R']
    plt.figure(figsize=(8,5))
    plt.plot(years, S, label='Susceptible')
    plt.plot(years, I, label='Infected')
    plt.plot(years, R, label='Recovered')
    plt.xlabel('Year')
    plt.ylabel('Number of Agents')
    plt.title('SIR Dynamics (Simple ABM)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Basic summary statistics
    final_S, final_I, final_R = S[-1], I[-1], R[-1]
    print(f"Final counts after {years[-1]} years: S={final_S}, I={final_I}, R={final_R}")

if __name__ == '__main__':
    run_demo()


## Interpretation of outputs

- The notebook prints a time series of S, I, and R counts over the simulated years.
- The final line summarizes the state of the population after the last year.
- You can adjust parameters at the top of the code cell that defines SimpleSIRABM to explore different dynamics:
  - Increase beta to simulate higher transmissibility
  - Increase gamma to simulate faster recovery
  - Increase n_agents to study larger populations (may impact runtime)
  - Toggle seed for reproducibility

Note: This is a compact, educational implementation and does not claim to reproduce Starsim's full capabilities such as multi-pathogen interactions, networked contacts, or demographic processes. It serves as a runnable starting point with clear structure for extension.